In [ ]:
# In[1]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[1]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[2]:

import pandas as pd

# Load files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
for df in (metric_df, trace_df, log_df, error_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Ensure numeric 'value' columns where present
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Helper to produce group summary for normalized files
def summarize_normalized(df, name_col):
    grp = df.groupby(['cmdb_id', name_col]).agg(
        total_count = ('value', 'size'),
        global_p95 = ('value', lambda s: s.quantile(0.95) if s.size>0 else pd.NA),
        min_timestamp = ('timestamp', 'min'),
        max_timestamp = ('timestamp', 'max')
    ).reset_index()
    grp = grp.rename(columns={'cmdb_id':'component', name_col:'kpi_name'})
    grp = grp.sort_values('total_count', ascending=False).head(50)
    # Keep compact column order
    return grp[['component','kpi_name','global_p95','total_count','min_timestamp','max_timestamp']]

# Produce summaries
metric_summary = summarize_normalized(metric_df, 'kpi_name')
trace_summary  = summarize_normalized(trace_df, 'trace_name')
log_summary    = summarize_normalized(log_df, 'log_name')

# Error logs summary: group by cmdb_id, sample up to 5 messages
def summarize_error_logs(df):
    def sample_msgs(s):
        # take first up to 5 messages in original ordering
        msgs = s.head(5).astype(str).tolist()
        return " || ".join(msgs)
    grp = df.groupby('cmdb_id').agg(
        total_count = ('message', 'size'),
        min_timestamp = ('timestamp','min'),
        max_timestamp = ('timestamp','max'),
        sample_messages = ('message', sample_msgs)
    ).reset_index().rename(columns={'cmdb_id':'component'})
    grp = grp.sort_values('total_count', ascending=False).head(50)
    return grp[['component','total_count','min_timestamp','max_timestamp','sample_messages']]

error_summary = summarize_error_logs(error_df)

# Display the four summaries (compact)
metric_summary, trace_summary, log_summary, error_summary

```
Out[2]:
```
summary = (
    "Summary of telemetry summaries (top groups per file):\n"
    "- Files loaded: metric.csv, trace.csv, log.csv, error_logs.csv. Error logs file contains no entries (no rows).\n"
    "- Data coverage (most top groups): timestamps range from 2024-01-24 16:50:00 UTC to 2024-01-24 17:14:00 UTC with 25 samples per top group.\n"
    "- Metrics (metric.csv): many groups have 25 samples. Notable p95 values: ts-admin-basic-info-service.diskio p95 ≈ 116,465; ts-travel-mongo.diskio and others also show very large diskio p95 values; ts-voucher-service.socket p95 = 4.0; ts-admin-basic-info-service.cpu p95 ≈ 1.59.\n"
    "- Traces (trace.csv): top trace groups also have 25 samples. Example: ts-user-service trace.to_ts-user-service.row_count p95 = 21.6. Many trace groups show sub-second latencies (p95 values in small decimals) and some row_count traces ~22.\n"
    "- Logs (log.csv): top log groups show log.row_count p95 values (e.g., ~33.8 for ts-admin-basic-info-service) and most log.error_count p95 = 0 for top services.\n"
    "- Overall interpretation: there are high disk I/O p95 values on several components (e.g., ts-admin-basic-info-service, ts-travel-mongo), socket p95 elevated for ts-voucher-service, but log.error_count is zero for top services and error_logs is empty. These indicate heavy disk I/O and some socket activity as the most prominent signals in the available telemetry; no error log messages were present to corroborate application-level errors."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                       component    kpi_name     global_p95  total_count             min_timestamp             max_timestamp
365           ts-voucher-service      socket       4.000000           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
0    ts-admin-basic-info-service         cpu       1.588456           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
1    ts-admin-basic-info-service      diskio  116464.690247           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
2    ts-admin-basic-info-service       error       0.000000           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
3    ts-admin-basic-info-service  latency-50       0.016923           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
..                           ...         ...            ...          ...                       ...                       ...
309              ts-travel-mongo      diskio  432139.914067           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
70               ts-config-mongo      socket       7.000000           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
71             ts-config-service         cpu       1.432790           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
72             ts-config-service      diskio  160483.081293           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
73             ts-config-service  latency-50       0.003937           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00

[50 rows x 6 columns],                        component                                           kpi_name  global_p95  total_count             min_timestamp             max_timestamp
783              ts-user-service                 trace.to_ts-user-service.row_count   21.600000           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
0                           root  trace.to_ts-admin-basic-info-service.duration_...    0.010701           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
1                           root  trace.to_ts-admin-basic-info-service.duration_p95    0.015481           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
2                           root    trace.to_ts-admin-basic-info-service.error_rate    0.000000           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
3                           root     trace.to_ts-admin-basic-info-service.row_count   22.000000           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
..                           ...                                                ...         ...          ...                       ...                       ...
73   ts-admin-basic-info-service  trace.to_ts-admin-basic-info-service.duration_p95    0.009876           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
74   ts-admin-basic-info-service    trace.to_ts-admin-basic-info-service.error_rate    0.000000           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
75   ts-admin-basic-info-service     trace.to_ts-admin-basic-info-service.row_count   22.000000           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
76   ts-admin-basic-info-service           trace.to_ts-config-service.duration_mean    0.003680           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
77   ts-admin-basic-info-service            trace.to_ts-config-service.duration_p95    0.006696           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00

[50 rows x 6 columns],                       component         kpi_name  global_p95  total_count             min_timestamp             max_timestamp
0   ts-admin-basic-info-service  log.error_count         0.0           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
1   ts-admin-basic-info-service    log.row_count        33.8           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
2       ts-admin-travel-service  log.error_count         0.0           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
3       ts-admin-travel-service    log.row_count        33.0           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
7          ts-assurance-service    log.row_count        16.4           25 2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00
..                          ...              ...         ...          ...                       ...                       ...
18       ts-consign-price-mongo  log.error_count         0.0           24 2024-01-24 16:50:00+00:00 2024-01-24 17:13:00+00:00
14              ts-config-mongo  log.error_count         0.0           24 2024-01-24 16:50:00+00:00 2024-01-24 17:13:00+00:00
15              ts-config-mongo    log.row_count         1.0           24 2024-01-24 16:50:00+00:00 2024-01-24 17:13:00+00:00
16             ts-consign-mongo  log.error_count         0.0           24 2024-01-24 16:50:00+00:00 2024-01-24 17:13:00+00:00
17             ts-consign-mongo    log.row_count         1.0           24 2024-01-24 16:50:00+00:00 2024-01-24 17:13:00+00:00

[50 rows x 6 columns], Empty DataFrame
Columns: [component, total_count, min_timestamp, max_timestamp, sample_messages]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present; otherwise load
try:
    metric_df
    trace_df
    log_df
    error_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    trace_df = pd.read_csv("trace.csv")
    log_df = pd.read_csv("log.csv")
    error_df = pd.read_csv("error_logs.csv")

# Ensure timestamps are parsed as UTC datetimes (Unix seconds)
for df in (metric_df, trace_df, log_df, error_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Ensure value columns numeric where applicable
for df in (metric_df, trace_df, log_df):
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Candidate services and incident window
services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
window_start = pd.to_datetime("2024-01-24 16:47:39", utc=True)
window_end   = pd.to_datetime("2024-01-24 17:17:39", utc=True)

def analyze_file(df, kpi_col):
    # Compute global p95 per (cmdb_id, kpi) using full series (rule 9)
    grp_all = df.groupby(['cmdb_id', kpi_col])['value'].agg(
        global_p95 = lambda s: float(s.dropna().quantile(0.95)) if s.dropna().size>0 else np.nan,
    ).reset_index()
    grp_all = grp_all.rename(columns={'cmdb_id':'component', kpi_col:'kpi_name'})
    # Filter to only candidate services groups (but keep groups even if no window rows)
    grp_all = grp_all[grp_all['component'].isin(services)].copy()
    if grp_all.empty:
        # Return empty compact table with expected columns
        cols = ['component','kpi_name','global_p95','total_count_in_window','anomalies_count_in_window',
                'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window','max_value_in_window']
        return pd.DataFrame(columns=cols)
    # Window rows for candidate services
    window_rows = df[(df['timestamp']>=window_start) & (df['timestamp']<=window_end) & (df['cmdb_id'].isin(services))].copy()
    # Rename KPI col in window rows for merging
    window_rows = window_rows.rename(columns={'cmdb_id':'component', kpi_col:'kpi_name'})
    # Merge global p95 into window rows to compute anomalies
    merged = window_rows.merge(grp_all, on=['component','kpi_name'], how='left')
    # Determine anomaly flag (value > global_p95). If global_p95 is NaN, anomaly cannot be determined -> False
    merged['is_anomaly'] = False
    # Only mark anomaly where global_p95 is finite and value > global_p95 and value not null
    mask = merged['global_p95'].notna() & merged['value'].notna() & (merged['value'] > merged['global_p95'])
    merged.loc[mask, 'is_anomaly'] = True
    # Aggregates per group within window
    if not merged.empty:
        agg_window = merged.groupby(['component','kpi_name']).agg(
            total_count_in_window = ('value','size'),
            anomalies_count_in_window = ('is_anomaly','sum'),
            max_value_in_window = ('value','max'),
        ).reset_index()
        # earliest/latest anomaly timestamps
        anom_rows = merged[merged['is_anomaly']].copy()
        if not anom_rows.empty:
            anom_ts = anom_rows.groupby(['component','kpi_name']).agg(
                earliest_anomaly_timestamp_in_window = ('timestamp','min'),
                latest_anomaly_timestamp_in_window = ('timestamp','max'),
            ).reset_index()
        else:
            anom_ts = pd.DataFrame(columns=['component','kpi_name',
                                            'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])
    else:
        # No window rows at all
        agg_window = pd.DataFrame(columns=['component','kpi_name','total_count_in_window','anomalies_count_in_window','max_value_in_window'])
        anom_ts = pd.DataFrame(columns=['component','kpi_name',
                                        'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window'])
    # Combine grp_all with window aggregates (left join so groups with no window rows are included)
    result = grp_all.merge(agg_window, on=['component','kpi_name'], how='left')
    result = result.merge(anom_ts, on=['component','kpi_name'], how='left')
    # Fill missing counts with 0 and other fields with NA as specified
    result['total_count_in_window'] = result['total_count_in_window'].fillna(0).astype(int)
    result['anomalies_count_in_window'] = result['anomalies_count_in_window'].fillna(0).astype(int)
    # For timestamp and max_value NaNs leave as NaT / NaN (will display as NA)
    # Reorder and select columns per requirement
    result = result[['component','kpi_name','global_p95','total_count_in_window','anomalies_count_in_window',
                     'earliest_anomaly_timestamp_in_window','latest_anomaly_timestamp_in_window','max_value_in_window']]
    # Sort by anomalies_count_in_window desc then total_count_in_window desc, limit 20
    result = result.sort_values(['anomalies_count_in_window','total_count_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)
    return result

metric_result = analyze_file(metric_df, 'kpi_name')
trace_result  = analyze_file(trace_df, 'trace_name')
log_result    = analyze_file(log_df, 'log_name')

# Determine whether NONE of the five services show any anomalies across all three files
total_anomalies = metric_result['anomalies_count_in_window'].sum() + trace_result['anomalies_count_in_window'].sum() + log_result['anomalies_count_in_window'].sum()
if total_anomalies == 0:
    anomaly_summary = "NONE of the five services show any anomalies across all three files."
else:
    anomaly_summary = f"Some anomalies detected: total_anomalies_in_window = {int(total_anomalies)} across the three files."

# Display compact tables and summary
metric_result, trace_result, log_result, anomaly_summary

```
Out[4]:
```
Summary (plain English):

- Overall: anomalies were detected. Total anomalies within the incident window across the three files = 88. So the answer is NOT "NONE".

Per-service findings (five candidates):

- ts-auth-service
  - Metrics: multiple KPIs (cpu, diskio, latency-50, latency-90, mem, socket, workload) had anomalies (each reported anomalies_count_in_window = 2 in the results). Metric anomalies occurred roughly between ~2024-01-24T16:50:00Z and ~2024-01-24T17:11:00Z depending on KPI.
  - Traces: several trace KPIs (duration_mean, duration_p95, row_count, etc.) showed anomalies (each with anomalies_count_in_window = 2). Trace anomaly timestamps span roughly 2024-01-24T16:52:00Z–2024-01-24T17:09:00Z.
  - Logs: log.row_count showed anomalies (anomalies_count_in_window = 2; earliest ~2024-01-24T16:57:00Z, latest ~2024-01-24T17:03:00Z). log.error_count remained 0.

- ts-order-service
  - Metrics: several KPIs (cpu, diskio, latency-50, latency-90, mem, socket, workload) had anomalies (each anomalies_count_in_window = 2). Metric anomaly timestamps span roughly 2024-01-24T16:53:00Z–2024-01-24T17:14:00Z depending on KPI.
  - Traces: multiple trace KPIs showed anomalies (each anomalies_count_in_window = 2), with trace anomaly timestamps around 2024-01-24T16:50:00Z–2024-01-24T17:14:00Z for different trace names.
  - Logs: log.row_count had anomalies (anomalies_count_in_window = 2; earliest ~2024-01-24T16:53:00Z, latest ~2024-01-24T16:59:00Z). log.error_count = 0.

- ts-route-service
  - Metrics: several KPIs (cpu, diskio, latency-50, latency-90, mem, socket) showed anomalies (anomalies_count_in_window = 2 per KPI). Metric anomaly timestamps roughly 2024-01-24T16:50:00Z–2024-01-24T17:14:00Z depending on KPI.
  - Traces: not prominent in the top trace rows returned for anomalies, but metrics and logs show anomalies.
  - Logs: log.row_count had anomalies (anomalies_count_in_window = 2; earliest ~2024-01-24T16:53:00Z, latest ~2024-01-24T16:59:00Z). log.error_count = 0.

- ts-travel-service
  - Metrics/traces: not shown as having anomalies in the returned top groups.
  - Logs: log.row_count shows anomalies (anomalies_count_in_window = 2; earliest ~2024-01-24T16:53:00Z, latest ~2024-01-24T16:59:00Z). log.error_count = 0.

- ts-train-service
  - No anomalies reported and no rows for these KPIs appeared in the compact results (no window anomalies observed for this service).

Additional notes:
- The anomalies are small in per-KPI counts in the window (typical anomalies_count_in_window = 2 for affected KPIs) but are present across many KPIs for the same services, producing the aggregate total of 88 anomalies.
- log.error_count is zero for the candidate services (no error log spikes), and error_logs.csv had no sampled error messages in the earlier summary.
- Conclusion: Multiple of the five services show anomalies during the incident window (notably ts-auth-service and ts-order-service across metrics, traces and logs; ts-route-service across metrics and logs; ts-travel-service in logs). Only ts-train-service showed no anomalies in the returned results.

The original code execution output of IPython Kernel is also provided below for reference:

(           component    kpi_name    global_p95  total_count_in_window  anomalies_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  max_value_in_window
0    ts-auth-service         cpu  1.514478e+01                     25                          2            2024-01-24 16:54:00+00:00          2024-01-24 16:58:00+00:00         1.802116e+01
1    ts-auth-service      diskio  1.146091e+05                     25                          2            2024-01-24 17:09:00+00:00          2024-01-24 17:11:00+00:00         1.189352e+05
2    ts-auth-service  latency-50  2.617599e-01                     25                          2            2024-01-24 16:53:00+00:00          2024-01-24 16:57:00+00:00         2.990701e-01
3    ts-auth-service  latency-90  8.732156e-01                     25                          2            2024-01-24 16:53:00+00:00          2024-01-24 16:54:00+00:00         8.990426e-01
4    ts-auth-service         mem  2.437057e+08                     25                          2            2024-01-24 16:54:00+00:00          2024-01-24 17:10:00+00:00         2.437927e+08
5    ts-auth-service      socket  1.414333e+01                     25                          2            2024-01-24 16:50:00+00:00          2024-01-24 16:53:00+00:00         1.447619e+01
6    ts-auth-service    workload  2.927200e+00                     25                          2            2024-01-24 16:57:00+00:00          2024-01-24 16:58:00+00:00         2.956667e+00
7   ts-order-service         cpu  2.321664e+00                     25                          2            2024-01-24 16:53:00+00:00          2024-01-24 17:00:00+00:00         2.364421e+00
8   ts-order-service      diskio  1.780996e+05                     25                          2            2024-01-24 17:07:00+00:00          2024-01-24 17:09:00+00:00         2.005939e+05
9   ts-order-service  latency-50  1.119243e-02                     25                          2            2024-01-24 16:55:00+00:00          2024-01-24 17:08:00+00:00         1.379486e-02
10  ts-order-service  latency-90  1.118880e-01                     25                          2            2024-01-24 16:56:00+00:00          2024-01-24 17:11:00+00:00         2.326201e-01
11  ts-order-service         mem  2.669203e+08                     25                          2            2024-01-24 17:10:00+00:00          2024-01-24 17:14:00+00:00         2.670283e+08
12  ts-order-service      socket  1.403000e+01                     25                          2            2024-01-24 17:08:00+00:00          2024-01-24 17:14:00+00:00         1.430000e+01
13  ts-order-service    workload  7.108900e+00                     25                          2            2024-01-24 17:09:00+00:00          2024-01-24 17:10:00+00:00         7.355467e+00
14  ts-route-service         cpu  4.574961e+00                     25                          2            2024-01-24 16:54:00+00:00          2024-01-24 17:06:00+00:00         4.592602e+00
15  ts-route-service      diskio  5.985429e+04                     25                          2            2024-01-24 17:02:00+00:00          2024-01-24 17:09:00+00:00         6.169640e+04
16  ts-route-service  latency-50  3.396560e-03                     25                          2            2024-01-24 16:50:00+00:00          2024-01-24 16:51:00+00:00         3.497870e-03
17  ts-route-service  latency-90  7.152685e-03                     25                          2            2024-01-24 16:50:00+00:00          2024-01-24 16:51:00+00:00         7.622659e-03
18  ts-route-service         mem  2.571562e+08                     25                          2            2024-01-24 17:11:00+00:00          2024-01-24 17:14:00+00:00         2.571985e+08
19  ts-route-service      socket  1.882667e+01                     25                          2            2024-01-24 16:53:00+00:00          2024-01-24 17:00:00+00:00         1.883333e+01,            component                                           kpi_name  global_p95  total_count_in_window  anomalies_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  max_value_in_window
0    ts-auth-service                      trace.from_root.duration_mean    0.360510                     25                          2            2024-01-24 16:52:00+00:00          2024-01-24 17:06:00+00:00             0.369631
1    ts-auth-service                       trace.from_root.duration_p95    0.970554                     25                          2            2024-01-24 16:53:00+00:00          2024-01-24 16:55:00+00:00             1.035617
2    ts-auth-service           trace.from_ts-auth-service.duration_mean    0.004252                     25                          2            2024-01-24 16:59:00+00:00          2024-01-24 17:09:00+00:00             0.004609
3    ts-auth-service            trace.from_ts-auth-service.duration_p95    0.007069                     25                          2            2024-01-24 16:57:00+00:00          2024-01-24 16:59:00+00:00             0.015319
4    ts-auth-service               trace.from_ts-auth-service.row_count  347.600000                     25                          2            2024-01-24 16:57:00+00:00          2024-01-24 17:03:00+00:00           360.000000
5    ts-auth-service             trace.to_ts-auth-service.duration_mean    0.004252                     25                          2            2024-01-24 16:59:00+00:00          2024-01-24 17:09:00+00:00             0.004609
6    ts-auth-service              trace.to_ts-auth-service.duration_p95    0.007069                     25                          2            2024-01-24 16:57:00+00:00          2024-01-24 16:59:00+00:00             0.015319
7    ts-auth-service                 trace.to_ts-auth-service.row_count  347.600000                     25                          2            2024-01-24 16:57:00+00:00          2024-01-24 17:03:00+00:00           360.000000
8   ts-order-service                      trace.from_root.duration_mean    0.626802                     25                          2            2024-01-24 17:13:00+00:00          2024-01-24 17:14:00+00:00             0.633859
9   ts-order-service                       trace.from_root.duration_p95    0.666899                     25                          2            2024-01-24 17:13:00+00:00          2024-01-24 17:14:00+00:00             0.673995
10  ts-order-service  trace.from_ts-inside-payment-service.duration_...    0.011903                     25                          2            2024-01-24 16:50:00+00:00          2024-01-24 16:51:00+00:00             0.013707
11  ts-order-service  trace.from_ts-inside-payment-service.duration_p95    0.019910                     25                          2            2024-01-24 16:50:00+00:00          2024-01-24 16:51:00+00:00             0.043416
12  ts-order-service          trace.from_ts-order-service.duration_mean    0.020708                     25                          2            2024-01-24 16:55:00+00:00          2024-01-24 17:11:00+00:00             0.028136
13  ts-order-service           trace.from_ts-order-service.duration_p95    0.014103                     25                          2            2024-01-24 16:55:00+00:00          2024-01-24 17:11:00+00:00             0.079516
14  ts-order-service              trace.from_ts-order-service.row_count  586.400000                     25                          2            2024-01-24 16:53:00+00:00          2024-01-24 16:59:00+00:00           617.000000
15  ts-order-service       trace.from_ts-preserve-service.duration_mean    0.017084                     25                          2            2024-01-24 16:55:00+00:00          2024-01-24 17:11:00+00:00             0.028192
16  ts-order-service        trace.from_ts-preserve-service.duration_p95    0.024438                     25                          2            2024-01-24 16:54:00+00:00          2024-01-24 17:11:00+00:00             0.028192
17  ts-order-service           trace.from_ts-preserve-service.row_count    6.800000                     25                          2            2024-01-24 16:59:00+00:00          2024-01-24 17:08:00+00:00             8.000000
18  ts-order-service           trace.from_ts-seat-service.duration_mean    0.008028                     25                          2            2024-01-24 17:08:00+00:00          2024-01-24 17:12:00+00:00             0.010181
19  ts-order-service            trace.from_ts-seat-service.duration_p95    0.018982                     25                          2            2024-01-24 17:08:00+00:00          2024-01-24 17:12:00+00:00             0.020298,            component         kpi_name  global_p95  total_count_in_window  anomalies_count_in_window earliest_anomaly_timestamp_in_window latest_anomaly_timestamp_in_window  max_value_in_window
0    ts-auth-service    log.row_count       347.6                     25                          2            2024-01-24 16:57:00+00:00          2024-01-24 17:03:00+00:00                  360
1   ts-order-service    log.row_count       504.2                     25                          2            2024-01-24 16:53:00+00:00          2024-01-24 16:59:00+00:00                  516
2   ts-route-service    log.row_count      1097.6                     25                          2            2024-01-24 16:53:00+00:00          2024-01-24 16:59:00+00:00                 1140
3  ts-travel-service    log.row_count      2164.8                     25                          2            2024-01-24 16:53:00+00:00          2024-01-24 16:59:00+00:00                 2236
4    ts-auth-service  log.error_count         0.0                     25                          0                                  NaT                                NaT                    0
5   ts-order-service  log.error_count         0.0                     25                          0                                  NaT                                NaT                    0
6   ts-route-service  log.error_count         0.0                     25                          0                                  NaT                                NaT                    0
7  ts-travel-service  log.error_count         0.0                     25                          0                                  NaT                                NaT                    0, 'Some anomalies detected: total_anomalies_in_window = 88 across the three files.')```
```

In [ ]:
# In[6]:

import pandas as pd
import re
import numpy as np

# Reuse trace_df if present, else load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# 1) Parse timestamp as UTC datetimes
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure numeric value
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Candidate services
services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 2) Compute global_p95 per (cmdb_id, trace_name) using full series
grp_p95 = trace_df.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})

# Merge global_p95 back to full trace_df
trace_full = trace_df.merge(grp_p95, on=['cmdb_id','trace_name'], how='left')

# 3) Filter rows to incident window (inclusive)
window_start = pd.to_datetime("2024-01-24 16:47:39", utc=True)
window_end   = pd.to_datetime("2024-01-24 17:17:39", utc=True)
trace_win = trace_full[(trace_full['timestamp'] >= window_start) & (trace_full['timestamp'] <= window_end)].copy()

# 4) Keep rows where cmdb_id is in services OR trace_name contains any of those service substrings
service_pattern = "(" + "|".join(re.escape(s) for s in services) + ")"
mask_service = trace_win['cmdb_id'].isin(services) | trace_win['trace_name'].str.contains(service_pattern, na=False)
trace_win = trace_win[mask_service].copy()

# 5) Mark anomalous rows where value > global_p95 (global_p95 computed from full series)
trace_win['is_anomaly'] = False
mask_anom = trace_win['value'].notna() & trace_win['global_p95'].notna() & (trace_win['value'] > trace_win['global_p95'])
trace_win.loc[mask_anom, 'is_anomaly'] = True

# Keep only anomalous rows for subsequent processing
anom = trace_win[trace_win['is_anomaly']].copy()

# If no anomalous rows found, return explicit message
if anom.shape[0] == 0:
    result = "Empty result: no anomalous rows found in the incident window for the requested filters."
else:
    # 6) Extract direction and referenced_service
    def extract_direction(name):
        if pd.isna(name):
            return 'other'
        if 'to_ts-' in name:
            return 'to'
        if 'from_ts-' in name:
            return 'from'
        if ('from_root' in name) or ('to_root' in name) or ('from_root' in name) or ('to_root' in name):
            return 'root'
        return 'other'

    def extract_referenced_service(name):
        if pd.isna(name):
            return pd.NA
        m = re.search(r'(?:to_ts-|from_ts-)([^\.]+)', name)
        if m:
            return m.group(1)
        return pd.NA

    anom['direction'] = anom['trace_name'].apply(extract_direction)
    anom['referenced_service'] = anom['trace_name'].apply(extract_referenced_service)
    # Treat NA referenced_service as string 'NA' to group
    anom['referenced_service'] = anom['referenced_service'].fillna('NA')

    # 7) Aggregate anomalous rows by referenced_service
    def first_n_list(seq, n=10):
        # preserve order of appearance
        seen = []
        for x in seq:
            if x not in seen:
                seen.append(x)
            if len(seen) >= n:
                break
        return seen

    agg = anom.groupby('referenced_service').agg(
        anomalies_count = ('is_anomaly','sum'),
        directions_present = ('direction', lambda s: sorted(set(s.tolist()))),
        earliest_anomaly_timestamp = ('timestamp','min'),
        latest_anomaly_timestamp = ('timestamp','max'),
        originating_cmdb_ids = ('cmdb_id', lambda s: first_n_list(s.tolist(), n=10)),
        max_value_in_window = ('value','max'),
        total_anomalous_rows = ('is_anomaly','sum')
    ).reset_index()

    # 8) Keep groups with anomalies_count > 0 (they should all be >0), sort and limit to 20
    agg = agg[agg['anomalies_count'] > 0].copy()
    agg = agg.sort_values(['anomalies_count','earliest_anomaly_timestamp'], ascending=[False, True]).head(20).reset_index(drop=True)

    # Ensure column order as requested
    result = agg[['referenced_service','anomalies_count','directions_present',
                  'earliest_anomaly_timestamp','latest_anomaly_timestamp',
                  'originating_cmdb_ids','max_value_in_window','total_anomalous_rows']]

# Display the result variable (DataFrame or message)
result

```
Out[6]:
```
Summary (plain English):

- There are many anomalous trace rows during the incident window. The anomalies are grouped by the referenced downstream/upstream service extracted from trace names.

- Top affected referenced services (by anomalous trace rows):
  1. travel-service — 70 anomalies (directions: from, to). Earliest anomaly 2024-01-24T16:51:00Z, latest 2024-01-24T17:14:00Z. Originating CMDBs include ts-route-service, root, ts-food-service, ts-travel-service, etc. Max anomalous value ~2002.
  2. order-service — 51 anomalies (from, to). Earliest 2024-01-24T16:50:00Z, latest 2024-01-24T17:14:00Z. Originators include ts-inside-payment-service, ts-order-service, etc. Max ~617.
  3. route-service — 30 anomalies (from, to). Earliest 16:50Z, latest 17:14Z. Originators include ts-travel2-service, ts-route-service, ts-travel-service. Max ~2280.
  4. train-service — 30 anomalies (from, to). Earliest 16:53Z, latest 17:14Z. Originators include ts-basic-service, ts-train-service, ts-travel-service. Max ~2888.
  5. seat-service — 18 anomalies (from, to). Earliest 16:53Z, latest 17:13Z. Originators include ts-order-service, ts-travel-service. Max ~292.
  6. auth-service — 17 anomalies (from, to). Earliest 16:52Z, latest 17:09Z. Originators include root and ts-auth-service. Max ~360.
  7. NA group — 16 anomalies (direction: root only). Earliest 16:52Z, latest 17:14Z. Originators include ts-auth-service, ts-travel-service, ts-order-service. Max ~90.
  - Additional smaller groups (travel2-service, basic-service, preserve-service, food-service, ticketinfo-service, security-service, inside-payment-service, station-service, admin-travel-service) also show anomalous rows.

- Timing: anomalous traces span roughly from ~2024-01-24T16:50Z through ~2024-01-24T17:14Z.

- Interpretation / next steps: trace anomalies are concentrated around travel-service and order-service (largest counts), with many anomalies involving both directions (from/to). These services — and the services that call them (originating cmdb_ids listed) — are the highest-priority candidates for investigation.

The original code execution output of IPython Kernel is also provided below for reference:

referenced_service  anomalies_count directions_present earliest_anomaly_timestamp  latest_anomaly_timestamp                               originating_cmdb_ids  max_value_in_window  total_anomalous_rows
0           travel-service               70         [from, to]  2024-01-24 16:51:00+00:00 2024-01-24 17:14:00+00:00  [ts-route-service, root, ts-food-service, ts-o...          2002.000000                    70
1            order-service               51         [from, to]  2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00  [ts-inside-payment-service, ts-order-service, ...           617.000000                    51
2            route-service               30         [from, to]  2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00  [ts-travel2-service, ts-route-service, ts-trav...          2280.000000                    30
3            train-service               30         [from, to]  2024-01-24 16:53:00+00:00 2024-01-24 17:14:00+00:00  [ts-basic-service, ts-train-service, ts-travel...          2888.000000                    30
4             seat-service               18         [from, to]  2024-01-24 16:53:00+00:00 2024-01-24 17:13:00+00:00              [ts-order-service, ts-travel-service]           292.000000                    18
5             auth-service               17         [from, to]  2024-01-24 16:52:00+00:00 2024-01-24 17:09:00+00:00                            [root, ts-auth-service]           360.000000                    17
6                       NA               16             [root]  2024-01-24 16:52:00+00:00 2024-01-24 17:14:00+00:00  [ts-auth-service, ts-travel-service, ts-order-...            90.000000                    16
7          travel2-service               12             [from]  2024-01-24 16:50:00+00:00 2024-01-24 17:14:00+00:00               [ts-route-service, ts-train-service]           653.000000                    12
8            basic-service               12             [from]  2024-01-24 16:53:00+00:00 2024-01-24 17:14:00+00:00               [ts-route-service, ts-train-service]           162.000000                    12
9         preserve-service               12             [from]  2024-01-24 16:54:00+00:00 2024-01-24 17:13:00+00:00              [ts-order-service, ts-travel-service]             8.000000                    12
10            food-service                6             [from]  2024-01-24 16:53:00+00:00 2024-01-24 17:12:00+00:00                                [ts-travel-service]            33.000000                     6
11      ticketinfo-service                6               [to]  2024-01-24 16:53:00+00:00 2024-01-24 17:14:00+00:00                                [ts-travel-service]           419.000000                     6
12        security-service                6             [from]  2024-01-24 16:56:00+00:00 2024-01-24 17:14:00+00:00                                 [ts-order-service]            12.000000                     6
13  inside-payment-service                5             [from]  2024-01-24 16:50:00+00:00 2024-01-24 17:09:00+00:00                                 [ts-order-service]            12.000000                     5
14         station-service                5               [to]  2024-01-24 16:55:00+00:00 2024-01-24 17:14:00+00:00                                 [ts-order-service]            15.000000                     5
15    admin-travel-service                4             [from]  2024-01-24 17:08:00+00:00 2024-01-24 17:12:00+00:00                                [ts-travel-service]             0.437605                     4```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Candidate services and incident window
services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
window_start = pd.to_datetime("2024-01-24 16:47:39", utc=True)
window_end   = pd.to_datetime("2024-01-24 17:17:39", utc=True)

# 1) Parse timestamp as UTC datetimes
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure numeric value
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 2) Compute global P95 per (cmdb_id, kpi_name) using full series (do NOT filter by window)
grp_p95 = metric_df.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})

# Keep only candidate services' groups (we only need these)
grp_p95 = grp_p95[grp_p95['cmdb_id'].isin(services)].copy()

# Merge global_p95 back into full metric_df
metric_full = metric_df.merge(grp_p95, on=['cmdb_id','kpi_name'], how='left')

# 3) Filter rows to the incident window
metric_win = metric_full[(metric_full['timestamp'] >= window_start) & (metric_full['timestamp'] <= window_end)].copy()

# Also limit to candidate services (ensure inclusion)
metric_win = metric_win[metric_win['cmdb_id'].isin(services)].copy()

# 4) Mark anomalies where value > global_p95 (only where global_p95 is present)
metric_win['is_anomaly'] = False
mask_anom = metric_win['value'].notna() & metric_win['global_p95'].notna() & (metric_win['value'] > metric_win['global_p95'])
metric_win.loc[mask_anom, 'is_anomaly'] = True

# 5) Aggregate per (component, kpi_name)
agg_kpi = metric_win.groupby(['cmdb_id','kpi_name','global_p95'], dropna=False).agg(
    anomalies_count_in_window = ('is_anomaly','sum'),
    earliest_anomaly_timestamp_in_window = ('timestamp', lambda s: s[metric_win.loc[s.index, 'is_anomaly']].min() if s.size>0 else pd.NaT),
    max_value_in_window = ('value', lambda s: s[metric_win.loc[s.index, 'is_anomaly']].max() if s.size>0 else np.nan)
).reset_index()

# The lambda above is a bit complex due to groupby; simpler approach: compute per-group using filtered anomalies
anom_rows = metric_win[metric_win['is_anomaly']].copy()
if not anom_rows.empty:
    kpi_anom_agg = anom_rows.groupby(['cmdb_id','kpi_name']).agg(
        anomalies_count_in_window = ('is_anomaly','sum'),
        earliest_anomaly_timestamp_in_window = ('timestamp','min'),
        max_value_in_window = ('value','max')
    ).reset_index()
else:
    kpi_anom_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomalies_count_in_window','earliest_anomaly_timestamp_in_window','max_value_in_window'])

# Merge global_p95 for completeness
kpi_result = grp_p95.merge(kpi_anom_agg, on=['cmdb_id','kpi_name'], how='left')
# Fill zeros for anomalies_count where NaN -> 0
kpi_result['anomalies_count_in_window'] = kpi_result['anomalies_count_in_window'].fillna(0).astype(int)
# Keep earliest timestamps and max values as they are (NaT / NaN if none)
kpi_result['earliest_anomaly_timestamp_in_window'] = kpi_result['earliest_anomaly_timestamp_in_window']
kpi_result['max_value_in_window'] = kpi_result['max_value_in_window']

# severity_ratio = max_value_in_window / global_p95 (or NA if global_p95==0 or max_value_in_window is NaN)
def compute_severity(row):
    gp = row['global_p95']
    mv = row['max_value_in_window']
    if pd.isna(mv) or pd.isna(gp) or gp == 0:
        return np.nan
    return float(mv) / float(gp)

kpi_result['severity_ratio'] = kpi_result.apply(compute_severity, axis=1)

# Rename cmdb_id to component for output
kpi_result = kpi_result.rename(columns={'cmdb_id':'component'})

# 6) Per-component summary across KPIs
# Ensure all candidate components are included
components_df = pd.DataFrame({'component': services})

# For per-component aggregations, use kpi_result (which has global_p95 and anomaly counts)
# total_anomalies_across_kpis: sum of anomalies_count_in_window per component
comp_agg = kpi_result.groupby('component').agg(
    total_anomalies_across_kpis = ('anomalies_count_in_window','sum'),
    earliest_anomaly_overall = ('earliest_anomaly_timestamp_in_window','min'),
    max_severity_ratio_overall = ('severity_ratio','max')
).reset_index()

# Include components with no rows in metric (ensure they appear)
component_summary = components_df.merge(comp_agg, on='component', how='left')
# Fill missing totals with 0 and leave timestamps/ratios as NA
component_summary['total_anomalies_across_kpis'] = component_summary['total_anomalies_across_kpis'].fillna(0).astype(int)
component_summary['earliest_anomaly_overall'] = component_summary['earliest_anomaly_overall']
component_summary['max_severity_ratio_overall'] = component_summary['max_severity_ratio_overall']

# Prepare final per-KPI output: only rows with anomalies_count_in_window > 0
metric_kpi_result = kpi_result[kpi_result['anomalies_count_in_window'] > 0].copy()

# Sort by severity_ratio desc (NA last) then anomalies_count_in_window desc, limit 20
metric_kpi_result = metric_kpi_result.sort_values(['severity_ratio','anomalies_count_in_window'], ascending=[False,False], na_position='last').head(20)

# Select required columns and order
metric_kpi_result = metric_kpi_result[['component','kpi_name','global_p95','anomalies_count_in_window',
                                       'earliest_anomaly_timestamp_in_window','max_value_in_window','severity_ratio']].reset_index(drop=True)

# Compact per-component summary columns order
component_summary = component_summary[['component','total_anomalies_across_kpis','earliest_anomaly_overall','max_severity_ratio_overall']]

# Display results (compact)
metric_kpi_result, component_summary

```
Out[7]:
```
summary = (
    "Summary of metric anomalies for the five candidate services (window 2024-01-24 16:47:39–17:17:39 UTC):\n\n"
    "Method: timestamps parsed as UTC; global P95 per (component, kpi) computed on the full series; anomalies = rows in window with value > global P95.\n\n"
    "Top per-KPI findings (sorted by severity_ratio):\n"
    "- ts-order-service / latency-90: severity_ratio ≈ 2.08 (global_p95=0.1119), 2 anomalous samples, earliest anomaly 2024-01-24T16:56:00Z — largest relative deviation observed.\n"
    "- ts-order-service / latency-50: severity_ratio ≈ 1.23, 2 anomalous samples, earliest 2024-01-24T16:55:00Z.\n"
    "- ts-auth-service / cpu: severity_ratio ≈ 1.19, 2 anomalous samples, earliest 2024-01-24T16:54:00Z.\n"
    "- ts-auth-service / latency-50: severity_ratio ≈ 1.14, 2 anomalous samples, earliest 2024-01-24T16:53:00Z.\n"
    "- ts-order-service / diskio: severity_ratio ≈ 1.13, 2 anomalous samples, earliest 2024-01-24T17:07:00Z.\n"
    "Other anomalous KPIs exist across the five services (examples include diskio, cpu, socket, latency-50/90, workload), with severity_ratios mostly in the ~1.02–1.13 range and each showing 2 anomalous samples in the window.\n\n"
    "Per-component summary (total anomalies across KPIs, earliest anomaly, max severity ratio):\n"
    "- ts-auth-service: total_anomalies_across_kpis = 14, earliest_anomaly_overall = 2024-01-24T16:50:00Z, max_severity_ratio_overall ≈ 1.1899\n"
    "- ts-order-service: total_anomalies_across_kpis = 14, earliest_anomaly_overall = 2024-01-24T16:53:00Z, max_severity_ratio_overall ≈ 2.0790\n"
    "- ts-route-service: total_anomalies_across_kpis = 14, earliest_anomaly_overall = 2024-01-24T16:50:00Z, max_severity_ratio_overall ≈ 1.0657\n"
    "- ts-train-service: total_anomalies_across_kpis = 14, earliest_anomaly_overall = 2024-01-24T16:58:00Z, max_severity_ratio_overall ≈ 1.0514\n"
    "- ts-travel-service: total_anomalies_across_kpis = 14, earliest_anomaly_overall = 2024-01-24T16:54:00Z, max_severity_ratio_overall ≈ 1.1250\n\n"
    "Interpretation and next steps:\n"
    "- All five services show anomalies in the incident window; ts-order-service shows the largest relative spike (latency-90 ≈ 2× the global P95).\n"
    "- Many anomalies are small multiplicative deviations (~1.02–1.20), likely indicating brief spikes rather than massive sustained failures, except the ts-order-service latency-90 spike which is most concerning.\n"
    "- Recommend prioritizing investigation of ts-order-service latency (and its diskio), and ts-auth-service CPU/latency KPIs, then reviewing callers and traces around the earliest anomaly timestamps listed above.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(            component    kpi_name     global_p95  anomalies_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  severity_ratio
0    ts-order-service  latency-90       0.111888                          2            2024-01-24 16:56:00+00:00             0.232620        2.079043
1    ts-order-service  latency-50       0.011192                          2            2024-01-24 16:55:00+00:00             0.013795        1.232517
2     ts-auth-service         cpu      15.144778                          2            2024-01-24 16:54:00+00:00            18.021162        1.189926
3     ts-auth-service  latency-50       0.261760                          2            2024-01-24 16:53:00+00:00             0.299070        1.142536
4    ts-order-service      diskio  178099.628491                          2            2024-01-24 17:07:00+00:00        200593.926822        1.126302
5   ts-travel-service  latency-90       0.298963                          2            2024-01-24 17:05:00+00:00             0.336329        1.124985
6   ts-travel-service         cpu       3.532311                          2            2024-01-24 16:54:00+00:00             3.877335        1.097676
7   ts-travel-service      diskio  148301.867084                          2            2024-01-24 17:05:00+00:00        159646.834913        1.076499
8    ts-route-service  latency-90       0.007153                          2            2024-01-24 16:50:00+00:00             0.007623        1.065706
9    ts-train-service  latency-90       0.070972                          2            2024-01-24 17:08:00+00:00             0.074624        1.051447
10    ts-auth-service      diskio  114609.136552                          2            2024-01-24 17:09:00+00:00        118935.198098        1.037746
11   ts-train-service      socket      18.990000                          2            2024-01-24 17:08:00+00:00            19.700000        1.037388
12   ts-train-service  latency-50       0.008611                          2            2024-01-24 17:12:00+00:00             0.008914        1.035232
13   ts-order-service    workload       7.108900                          2            2024-01-24 17:09:00+00:00             7.355467        1.034684
14   ts-train-service         cpu      32.541809                          2            2024-01-24 17:09:00+00:00            33.654389        1.034189
15   ts-route-service      diskio   59854.289425                          2            2024-01-24 17:02:00+00:00         61696.399994        1.030777
16   ts-route-service  latency-50       0.003397                          2            2024-01-24 16:50:00+00:00             0.003498        1.029827
17    ts-auth-service  latency-90       0.873216                          2            2024-01-24 16:53:00+00:00             0.899043        1.029577
18    ts-auth-service      socket      14.143333                          2            2024-01-24 16:50:00+00:00            14.476190        1.023535
19  ts-travel-service      socket      18.686667                          2            2024-01-24 16:54:00+00:00            19.050000        1.019443,            component  total_anomalies_across_kpis  earliest_anomaly_overall  max_severity_ratio_overall
0    ts-auth-service                           14 2024-01-24 16:50:00+00:00                    1.189926
1   ts-order-service                           14 2024-01-24 16:53:00+00:00                    2.079043
2   ts-route-service                           14 2024-01-24 16:50:00+00:00                    1.065706
3   ts-train-service                           14 2024-01-24 16:58:00+00:00                    1.051447
4  ts-travel-service                           14 2024-01-24 16:54:00+00:00                    1.124985)```
```